In [ ]:
from collections import defaultdict

#parse context free grammar from string format to dict format
def parseCFG(cfg):
    #store nonterminals
    grammar = defaultdict(list)
    #splits input
    lines = cfg.strip().split("\n")
    
    #split right and left
    for line in lines:
        left, right = line.split("->")
        #remove whitespace
        left = left.strip()
        #separate productions by |
        productions = right.strip().split("|")
        for prod in productions:
            #remove whitespace
            prod = prod.strip()
            #empty
            if prod == "empty":
                grammar[left].append([])
            else:
                #store as a list
                grammar[left].append(list(prod))
    return grammar

def npda(grammar, string):
    #avoid infinite loops by limiting depth of search
    #AI was used for degugging from a runtime error (infinite loops)
    MAX = 10000

    def dfs(input_str, stack, depth, visited=None):
        if visited is None:
            # track visited states to avoid infinite loops
            visited = set()
            
        # limit depth to prevent infinite recursion
        if depth > MAX:
            return False
        
        #accept if stack is empty and input done
        if not stack:
            return not input_str
        
        #reject if stack is too big
        if len(stack) > len(input_str) * 3 + 10:
            return False

        #get top of stack and pop
        top = stack[-1]
        stack = stack[:-1]

        state = (input_str, top, tuple(stack))
        #avoid infinite loops
        if state in visited:
            return False
        #add current state to visited
        visited = visited | {state}

        #check if match terminal
        if top not in grammar:
            if input_str and input_str[0] == top:
                return dfs(input_str[1:], stack, depth + 1, visited)
            #reject if mismatch
            return False

        # try all productions for nonterminal
        for production in grammar[top]:
            #copy stack
            newStack = list(stack)
            for symbol in reversed(production):
                newStack.append(symbol)
            #continue recusively 
            if dfs(input_str, newStack, depth + 1, visited):
                return True
        #nothing worked, reject
        return False

    #start with S
    return dfs(string, ['S'], 0)

#run strings through npda and print results
def run(grammar, strings):
    for s in strings:
        result = npda(grammar, s)
        #print result
        print(f"{s} -> {'accept' if result else 'reject'}")

#Main: test cases 1-5
if __name__ == "__main__":
    print("\nTest Case 1:")
    cfg1 = """
    S -> AA
    A -> aAb | empty
    """
    g1 = parseCFG(cfg1)
    run(g1, ["aababb", "aaaaabbbbb"])

    print("\nTest Case 2:")
    cfg2 = """
    S -> U|V
    U -> XAX|UU
    V -> XBX|VV
    X -> aXb|bXa|XX|empty
    A -> aA|a
    B -> bB|b
    """
    g2 = parseCFG(cfg2)
    run(g2, ["baabba", "aaabbb"])

    print("\nTest Case 3:")
    cfg3 = """
    S -> UaU
    U -> bUaUa|aUbUa|aUaUb|UU|empty
    """
    g3 = parseCFG(cfg3)
    run(g3, ["bbaabaaaaa", "abaaba"])

    print("\nTest Case 4:")
    cfg4 = """
    S -> XaXaX
    X -> aXb|bXa|XX|empty
    """
    g4 = parseCFG(cfg4)
    run(g4, ["baabaa", "abaabab"])

    print("\nTest Case 5:")
    cfg5 = """
    S -> UV
    U -> XAX|UU
    X -> aXb|bXa|XX|empty
    A -> aA|a
    V -> aV|bV|empty
    """
    g5 = parseCFG(cfg5)
    run(g5, ["bbbaab", "aababbb"])
